In [1]:
"""
Agricultural Packaging Waste Demand Estimation via Artificial Neural Networks (ANN)
"""

# ==============================================================================
# IMPORTS & ENVIRONMENT SETUP
# ==============================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Visual configurations for academic plotting
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['font.family'] = 'serif'


In [2]:
# ==============================================================================
# 1. DATA INGESTION & PREPROCESSING
# ==============================================================================
print("--- STAGE 1: Data Ingestion ---")
try:
    df = pd.read_excel('Model_Waste.xlsx', header=0)
    print("Successfully loaded 'Model_Waste.xlsx'.")
except FileNotFoundError:
    raise FileNotFoundError("Error: 'Model_Waste.xlsx' not found in the directory.")

# Standardize column names by removing line breaks and trailing spaces
df.columns = df.columns.str.replace('\n', ' ').str.strip()

# Define feature variables (X) and target variable (Y)
# Note: Strings are kept in Portuguese to match the exact columns in the Excel file
target_col_original = 'Embalagens Tocantins'
feature_cols = ['Embalagens Brasil', 'Área Plantada Tocantins']

# Adjusting target variable to represent 100% of generated waste (94% collection rate assumption)
COLLECTION_RATE = 0.94
target_col_adjusted = 'Adjusted_Waste_TO_100'
df[target_col_adjusted] = df[target_col_original] / COLLECTION_RATE

# Data filtering: Exclude 2020 anomaly and remove missing values for training
df_train = df[df[target_col_adjusted].notna() & (df['Ano'] != 2020)].copy()

X_train_raw = df_train[feature_cols].values
Y_train_raw = df_train[target_col_adjusted].values

# ==============================================================================
# 2. FEATURE SCALING (STANDARDIZATION)
# ==============================================================================
# Standardization is strictly required for MLP gradient descent convergence (Mean=0, Std=1)
scaler_X = StandardScaler()
scaler_Y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train_raw)
Y_train_scaled = scaler_Y.fit_transform(Y_train_raw.reshape(-1, 1)).ravel()

# ==============================================================================
# 3. MULTILAYER PERCEPTRON (MLP) ARCHITECTURE & TRAINING
# ==============================================================================
# Architecture: 1 Hidden Layer (3 Neurons), Tanh activation function
mlp_model = MLPRegressor(
    hidden_layer_sizes=(3,),
    activation='tanh',
    solver='lbfgs',      # Quasi-Newton method, optimal for small datasets
    max_iter=10000,
    random_state=42,
    alpha=0.01           # L2 regularization term to mitigate overfitting
)

mlp_model.fit(X_train_scaled, Y_train_scaled)

# ==============================================================================
# 4. MODEL EVALUATION & SIGNIFICANCE
# ==============================================================================
# Generating predictions for evaluation
Y_pred_scaled = mlp_model.predict(X_train_scaled)
Y_pred_raw = scaler_Y.inverse_transform(Y_pred_scaled.reshape(-1, 1)).ravel()

# Calculating performance metrics
r2 = r2_score(Y_train_raw, Y_pred_raw)
mae = mean_absolute_error(Y_train_raw, Y_pred_raw)
rmse = np.sqrt(mean_squared_error(Y_train_raw, Y_pred_raw))

print("\n--- STAGE 4: Model Significance ---")
print(f"R-squared (Coefficient of Determination): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae/1e6:.2f} Million kg")
print(f"Root Mean Square Error (RMSE): {rmse/1e6:.2f} Million kg\n")

# ==============================================================================
# 5. MATHEMATICAL FORMULATION EXTRACTION
# ==============================================================================
# Extracting network weights and biases for external mathematical modeling
W1 = mlp_model.coefs_[0]       # Weights: Input to Hidden Layer
b1 = mlp_model.intercepts_[0]  # Biases: Hidden Layer
W2 = mlp_model.coefs_[1]       # Weights: Hidden Layer to Output
b2 = mlp_model.intercepts_[1]  # Biases: Output Layer

mu_X = scaler_X.mean_
sd_X = scaler_X.scale_
mu_Y = scaler_Y.mean_[0]
sd_Y = scaler_Y.scale_[0]

print("--- STAGE 5: Extracted Mathematical Formulation ---")
print("To calculate municipal generation, apply the following proportional downscaling method")
print("using the statewide neural network equations:\n")

print("STEP 1: Normalize input features for the target year:")
print(f"X1_norm (National_Waste) = (National_Waste_Input - {mu_X[0]:.2f}) / {sd_X[0]:.2f}")
print(f"X2_norm (State_Area) = (State_Area_Input - {mu_X[1]:.2f}) / {sd_X[1]:.2f}\n")

# CRITICAL FIX: The previous script incorrectly mentioned ReLU.
# The model is trained using 'tanh' (Hyperbolic Tangent).
print("STEP 2: Compute Hidden Layer Neurons (using Hyperbolic Tangent 'tanh' activation):")
for i in range(3):
    print(f"Neuron N{i+1} = tanh( ({W1[0, i]:.4f} * X1_norm) + ({W1[1, i]:.4f} * X2_norm) + ({b1[i]:.4f}) )")

print("\nSTEP 3: Compute Output Layer and Denormalize to obtain Total State Waste:")
print(f"Network_Sum = ({W2[0,0]:.4f} * N1) + ({W2[1,0]:.4f} * N2) + ({W2[2,0]:.4f} * N3) + ({b2[0]:.4f})")
print(f"Total_Waste_State (ANN) = (Network_Sum * {sd_Y:.2f}) + {mu_Y:.2f}\n")

print("STEP 4: Calculate Final Municipal Generation (Downscaling):")
print("Municipal_Generation (kg) = (Total_Waste_State / State_Area) * Municipal_Area")
print("=================================================================\n")

# ==============================================================================
# 6. MUNICIPAL DOWNSCALING STRATEGY
# ==============================================================================
print("--- STAGE 6: Municipal Downscaling Implementation ---")

try:
    # Load municipal data (assuming row 1 is the header)
    df_muni = pd.read_excel('Data_TO.xlsx', header=0)

    # Dynamically identify columns to prevent KeyErrors due to trailing spaces in Excel
    col_muni_name = df_muni.columns[0]
    col_muni_area = df_muni.columns[1]

    num_municipalities = len(df_muni)
    print(f"Successfully loaded 'Data_TO.xlsx' containing {num_municipalities} municipalities.")

    # Step 6.1: Aggregate municipal areas to establish the statewide macro-scenario
    total_state_area = df_muni[col_muni_area].sum()

    # Step 6.2: Retrieve the most recent national scenario (Embalagens Brasil) from the main dataset
    latest_year = df['Ano'].max()
    latest_national_waste = df.loc[df['Ano'] == latest_year, feature_cols[0]].values[0]

    print(f"-> Total Area extracted from municipal dataset: {total_state_area:,.2f} hectares")
    print(f"-> Employing national macro-scenario from the year {latest_year}.")

    # Step 6.3: Execute neural network prediction for the state total
    # Constructing input array: [National_Waste, Total_State_Area]
    macro_scenario_input = np.array([[latest_national_waste, total_state_area]])

    # Scaling, Prediction, and Inverse Scaling
    macro_scenario_scaled = scaler_X.transform(macro_scenario_input)
    predicted_state_waste_scaled = mlp_model.predict(macro_scenario_scaled)
    predicted_state_waste_raw = scaler_Y.inverse_transform(predicted_state_waste_scaled.reshape(-1, 1)).ravel()[0]

    # Step 6.4: Establish the Dynamic Generation Factor (kg/ha) derived from ANN
    dynamic_generation_factor = predicted_state_waste_raw / total_state_area
    print(f"-> Dynamic Generation Factor established by ANN: {dynamic_generation_factor:.4f} kg/ha\n")

    # Step 6.5: Compute individual generation per municipality
    col_generated_waste = 'Generated Waste (kg)'
    df_muni[col_generated_waste] = df_muni[col_muni_area] * dynamic_generation_factor

    # Rounding for presentation purposes
    df_muni[col_generated_waste] = df_muni[col_generated_waste].round(2)

    # Exporting finalized dataset
    output_filename = 'Data_TO_Calculated.xlsx'
    df_muni.to_excel(output_filename, index=False)

    print("--- DOWN-SCALED MUNICIPAL RESULTS (PREVIEW) ---")
    print(df_muni.head(5))
    print(f"\nProcessing successful. Results exported to: '{output_filename}'.")

except FileNotFoundError:
    print("CRITICAL ERROR: 'Data_TO.xlsx' was not found in the working directory.")
    print("Please ensure the spreadsheet is located in the same folder as this script to proceed with downscaling.")
except Exception as e:
    print(f"An unexpected error occurred during municipal data processing: {e}")

--- STAGE 1: Data Ingestion ---
Successfully loaded 'Model_Waste.xlsx'.

--- STAGE 4: Model Significance ---
R-squared (Coefficient of Determination): 0.9983
Mean Absolute Error (MAE): 0.01 Million kg
Root Mean Square Error (RMSE): 0.01 Million kg

--- STAGE 5: Extracted Mathematical Formulation ---
To calculate municipal generation, apply the following proportional downscaling method
using the statewide neural network equations:

STEP 1: Normalize input features for the target year:
X1_norm (National_Waste) = (National_Waste_Input - 47103763.38) / 4844361.92
X2_norm (State_Area) = (State_Area_Input - 1415976.88) / 422520.23

STEP 2: Compute Hidden Layer Neurons (using Hyperbolic Tangent 'tanh' activation):
Neuron N1 = tanh( (0.3357 * X1_norm) + (0.2977 * X2_norm) + (-0.7571) )
Neuron N2 = tanh( (0.6825 * X1_norm) + (-0.4457 * X2_norm) + (0.3667) )
Neuron N3 = tanh( (-0.3265 * X1_norm) + (1.5343 * X2_norm) + (0.4243) )

STEP 3: Compute Output Layer and Denormalize to obtain Total State